# TRABALHO PRÁTICO - PRÉ PROCESSAMENTO E ANÁLISE PREDITIVA

## Aluno: Luca Samuel dos Santos

## Professor: Prof. Douglas Castilho

## Matéria: Inteligência Artificial

---

## Base de Dados: Chronic Kidney Disease 

## Objetivo do trabalho: 

Realizar o pré-processamento e a análise preditiva sobre a base de dados de Doença Renal Crônica. Este relatório contém o passo a passo detalhado das etapas de um bom pré-processamento e de uma boa análise preditiva, descrevendo tudo o que foi pensado durante a realização do trabalho.

---

### 1. Identificação do atributo alvo (saída):

Na base de dados trabalhada, o atributo alvo é o atributo **class** podendo conter duas classes: *cdk* e *notcdk*

**Obs**: Base desbalanceada. Para uma base de 400 dados, 250 são *cdk* e 150 são *notcdk*

### 2,3. Identificação dos tipos e escala de dados dos atributos de entrada (quantitativo, qualitativo) e (nominal, ordinal, intervalar, racional)

age (Idade) - Quantitativo, Racional: Idade em Anos
bp (Pressão Sanguínea) - Quantitativo, Racional: Pressão em mg/Hg
sp (Gravidade Específica) - Qualitativo, Ordinal: Dividido entre as classes [1.005,1.010,1.015,1.020,1.025]
al (Albumina) - Qualitativo, Ordinal: Dividido entre as classes [0,1,2,3,4,5]
su (Açúcar) - Qualitativo, Ordinal: Dividido entre as classes [0,1,2,3,4,5]
rbc (Glóbulos Vermelhos) - Qualitativo, Nominal: Dividido entre as classes [normal,abnormal]
pc (Células de Pus) - Qualitativo, Nominal: Dividido entre as classes [normal,abnormal]
pcc (Aglomerados de Células de Pus) - Qualitativo, Nominal: Dividido entre as classes [present,notpresent]
ba (Bactéria) - Qualitativo, Nominal: Dividido entre as classes [present,notpresent]
bgr (Glicose Sanguínea Aleatória) - Quantitativo, Racional: Medido em mgs/dl
bu (Ureia no Sangue) - Quantitativo, Racional: Medido em mgs/dl
sc (Creatinina Sérica) - Quantitativo, Racional: Medido em mgs/dl
sod (Sódio) - Quantitativo, Racional: Medido em mEq/L
pot (Potássio) - Quantitativo, Racional: Medido em mEq/L
hemo (Hemoglobina) - Quantitativo, Racional: Medido em gms
pcv (Volume Globular/Hematócrito) - Quantitativo, Racional
wc (Contagem de Glóbulos Brancos) - Quantitativo, Racional: cells/cumm
rc (Contagem de Glóbulos Vermelhos) - Quantitativo, Racional: millions/cmm
htn (Hipertensão) - Qualitativo, Nominal: Dividido entre as classes [yes,no]
dm (Diabetes) - Qualitativo, Nominal: Dividido entre as classes [yes,no]
cad (Doença Arterial Coronariana) - Qualitativo, Nominal: Dividido entre as classes [yes,no]
appet (Apetite) - Qualitativo, Ordinal: Dividido entre as classes [good,poor]
pe (Edema Podal) - Qualitativo, Nominal: Dividido entre as classes [yes,no]
ane (Anemia) - Qualitativo, Nominal: Dividido entre as classes [yes,no]
class (Classe/Diagnóstico) - Qualitativo, Nominal: Dividido entre as classes [cdk,notcdk]

Código para armazenamento dos atributos e dados a partir do arquivo "chronic_kidney_disease":

In [23]:
def ler_arff(caminho_arquivo):
    atributos = []
    dados = []
    lendo_dados = False

    with open(caminho_arquivo, 'r', encoding='utf-8') as arquivo:
        for linha in arquivo:
            linha = linha.strip()
            
            if not linha or linha.startswith('%'):
                continue

            if linha.upper().startswith('@ATTRIBUTE'):
                partes = linha.split()
                if partes[1].startswith("'") and partes[1].endswith("'"):
                    partes[1] = partes[1][1:-1]
                nome_atributo = partes[1]
                atributos.append(nome_atributo)
            
            elif linha.upper().startswith('@DATA'):
                lendo_dados = True

            elif lendo_dados:
                valores = [valor.strip() for valor in linha.split(',')]
                dados.append(valores)

    return atributos, dados

In [24]:
caminho = './base_de_dados/chronic_kidney_disease.arff'

try:
    atributos, dados = ler_arff(caminho)

    print("ATRIBUTOS LIDOS:")
    print(atributos)

    for dado in dados:
        for i in range(len(dado)):
            if dado[i].isdigit():
                dado[i] = int(dado[i])
            elif dado[i].replace('.', '', 1).isdigit():
                dado[i] = float(dado[i])
        

    print("\nPRIMEIROS 5 DADOS LIDOS:")
    for i in range(min(5, len(dados))):
        print(dados[i])

    print(f"\nTotal de atributos: {len(atributos)}")
    print(f"Total de dados: {len(dados)}")
except FileNotFoundError:
    print(f"Erro: O arquivo '{caminho}' não foi encontrado.")

ATRIBUTOS LIDOS:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wbcc', 'rbcc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'class']

PRIMEIROS 5 DADOS LIDOS:
[48, 80, 1.02, 1, 0, '?', 'normal', 'notpresent', 'notpresent', 121, 36, 1.2, '?', '?', 15.4, 44, 7800, 5.2, 'yes', 'yes', 'no', 'good', 'no', 'no', 'ckd']
[7, 50, 1.02, 4, 0, '?', 'normal', 'notpresent', 'notpresent', '?', 18, 0.8, '?', '?', 11.3, 38, 6000, '?', 'no', 'no', 'no', 'good', 'no', 'no', 'ckd']
[62, 80, 1.01, 2, 3, 'normal', 'normal', 'notpresent', 'notpresent', 423, 53, 1.8, '?', '?', 9.6, 31, 7500, '?', 'no', 'yes', 'no', 'poor', 'no', 'yes', 'ckd']
[48, 70, 1.005, 4, 0, 'normal', 'abnormal', 'present', 'notpresent', 117, 56, 3.8, 111, 2.5, 11.2, 32, 6700, 3.9, 'yes', 'no', 'no', 'poor', 'yes', 'yes', 'ckd']
[51, 80, 1.01, 2, 0, 'normal', 'normal', 'notpresent', 'notpresent', 106, 26, 1.4, '?', '?', 11.6, 35, 7300, 4.6, 'no', 'no', 'no', 'good', 'no', 'no', '

### 4,5,6. Exploração dos dados através de medidas de localidade, espalhamento e distribuição

Código utilizado para encontrar as medidas de localidade, adotando que para dados simbólicos, será utilizado a moda e para atributos numéricos serão utilizados média, mediana e percentil: